# LLM setup



In [1]:
!pip uninstall -y datasets


Found existing installation: datasets 2.14.4
Uninstalling datasets-2.14.4:
  Successfully uninstalled datasets-2.14.4


In [4]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

IN_COLAB = "google.colab" in sys.modules


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

# Skipped once the packages are installed, so this cell can be re-run after restarting the session.
if IN_COLAB and installed_version("pandasai") != "3.0.0":
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError("Choose a Python 3.11 Colab runtime before installing these packages.")
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    # Same versions as uv.lock (torch is preinstalled on Colab).
    packages = [
        "ipython>=8,<9",
        "matplotlib-inline<0.2",
        "numpy==1.26.4",
        "pandas==2.3.3",
        "transformers==5.17.0",
        "accelerate==1.15.0",
        "bitsandbytes==0.50.2",
        "peft==0.21.2",
        "sentence-transformers==6.1.0",
        "requests",
        "pandasai==3.0.0",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Choose Runtime → Restart session before continuing.")

In [1]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 117.0 MB/s eta 0:00:00


## Get the project code from github

On Colab, clone your team's repository so that the `ift6758` package can be imported.
For a private repository, create a GitHub [personal access token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens) with read access and add it to Colab's *Secrets* (key icon in the left sidebar) as `GITHUB_TOKEN`. Never paste the token in the notebook.

In [5]:
import os
import subprocess
IN_COLAB=True
if IN_COLAB:
    from google.colab import userdata

    REPO = "aliciavaq/data_science_project"
    REPO_DIR = "/content/repo"
    if not os.path.exists(REPO_DIR):
        token = userdata.get("GITHUB_TOKEN")
        subprocess.check_call(["git", "clone", f"https://{token}@github.com/{REPO}.git", REPO_DIR])
    sys.path.insert(0, REPO_DIR)

In [ ]:
!git -C /content/repo pull

Already up to date.


## Check the GPU

In [6]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU found. On Colab, select a T4 GPU runtime (Runtime → Change runtime type).")

GPU: Tesla T4


## Load the model

The first run downloads ~8 GB of weights to the Hugging Face cache.

In [7]:
from ift6758.llm import load_model

model, tokenizer = load_model()
print(f"Loaded on {model.device} ({model.dtype})")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded on cuda:0 (torch.float16)


In [8]:
#test if model interface works
messages = [{"role": "user", "content": "In one sentence, what is a hat trick in hockey?"}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)
output_ids = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In hockey, a hat trick occurs when a player scores a goal, records an assist, and makes a save in the same game.


# Manual implementation of function to add new feature

In [9]:
import numpy as np
import pandas as pd



def add_shot_distance_manual(df,net_X = 89):
    #This function adds the shot distance and angle from the net.
    #Standard rink is 200 feet length total and net is 11 feet from wall

    df = df.copy()
     #

    # Assume  shot is aimed at the net on the same side of the rink as  shot.
    d_goal_line = net_X - df["x"].abs()

    # How far the shot is to the side of the middle of the net
    d_side = df["y"].abs()

    # Distance in feet from the shot to the center of the net
    df["shot_distance"] = np.sqrt(d_goal_line ** 2 + d_side ** 2)




    return df



In [12]:
import numpy as np
import pandas as pd
from ift6758.llm.qwen import ask_qwen
from IPython.display import  Markdown

code_prompt="""

TASK: Write a function `add_shot_distance_llm(df) -> pandas.DataFrame` that takes a pandas Dataframe of NHL game events
 and return a df with an extra feature called `shot_distance`,given by the distance in feet between the shot (x, y) and
 the center of the net.



SPECIFICATIONS:
1. The coordinates `x` and `y` are in feet with (0, 0) being the center position. The rink is 200 feet long and 85 feet wide.
2. The two nets are on the goal lines at 11 feet from each end wall.
3. A shot done closer to the net has a better chance to end in a goal. Compute the distance with the coordinates
   and the location of the goals, and use the net that the shot is aimed at.
4. Add the result in a new column named `shot_distance`, in feet.
5. Do not modify the input DataFrame and only add new feature
6. Use minimal comments


Return only the code in a single Python code block.
"""
generated_code = ask_qwen(model, tokenizer, code_prompt,
               system="You are an expert Python data engineer.",
               max_new_tokens=500)
display(Markdown(generated_code))


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```python
import pandas as pd
import numpy as np

def add_shot_distance_llm(df):
    df_copy = df.copy()
    # Goal centers are at (100, 0) for the left net and (100, 0) for the right net
    # But since the rink is 200 feet long, the center of the net is at (100, 0)
    # The net is 11 feet from each end wall, so the center of the net is at (100, 0)
    # For a shot at (x, y), distance to net center is sqrt((x - 100)^2 + y^2)
    df_copy['shot_distance'] = np.sqrt((df_copy['x'] - 100)**2 + df_copy['y']**2)
    return df_copy
```